# Building Mathematical Reasoning Systems

## Autoformalization Tasks

**Autoformalization** is the process of translating informal mathematical content (e.g., natural language or LaTeX) into a formal language that can be processed by a proof assistant such as **Isabelle** or **Lean**.

Autoformalization can be categorized into four tasks:

1. **Definition Autoformalization:** Translate informal mathematical definitions into formal definitions.

2. **Statement Autoformalization:** Translate informal mathematical statements (e.g., theorems and lemmas) into formal statements, without constructing their proofs.

3. **Proof Autoformalization:** Translate informal mathematical proofs into machine-checkable formal proofs, typically given an existing formal theorem statement.

4. **Full-Theorem Autoformalization:** Translate both an informal theorem statement and its proof into a complete formal theorem with a machine-checkable proof.

**Key distinction:** Definition autoformalization provides the *foundational building blocks* of formal mathematics. Statement autoformalization captures *what is being claimed*, while proof autoformalization captures *why the claim holds*. Full-theorem autoformalization addresses both.

A typical autoformalization workflow is illustrated below:

![Illustration](https://raw.githubusercontent.com/lanzhang128/multi_agent_autoformalization/main/illustration.png)

*Figure 1: Illustration of an LLM-based autoformalization workflow.*

As shown in Figure 1, the process begins with an informal mathematical statement expressed in natural language. An LLM first produces an informal proof or reasoning process, which provides additional context for formalization. Relevant mathematical definitions and theorems can then be retrieved from a knowledge base constructed from formal libraries.

Based on this information, an LLM generates candidate formalizations in the language of a theorem prover, such as Isabelle or Lean. These candidates can be further improved through **iterative refinement**, incorporating feedback from LLM-based critiques and theorem provers. A candidate selection process identifies an accepted formalization, which is subsequently submitted to a theorem prover for formal verification.

This workflow highlights the importance of combining **LLM-based generation, knowledge retrieval, iterative refinement, and formal verification** to improve the reliability of autoformalization.

In this tutorial, we provide a **hands-on demonstration** of how to build an LLM-based autoformalization system from individual components. We begin with **definition autoformalization**, introducing several key components step by step:

1. **Zero-shot Autoformalization:** Generate initial Lean formalizations directly from natural language descriptions using an LLM.
2. **Hard Critique from Lean:** Use feedback from the Lean theorem prover to identify syntax errors, type mismatches, and other formalization issues.
3. **Soft Feedback from LLM Judges:** Employ LLM-based judges to evaluate the semantic correctness and alignment of generated formalizations with their informal descriptions.
4. **Feedback-Guided Refinement:** Incorporate both hard feedback from Lean and soft feedback from LLM judges to refine candidate formalizations.

After demonstrating these components individually, we integrate them into a unified **agentic autoformalization workflow** for **theorem statement formalization** and **full-theorem autoformalization**. We also use Isabll/HOL to prove the cross-prover ability of the system. Through this step-by-step demonstration, participants will gain practical experience in constructing, evaluating, and refining LLM-based autoformalization systems.

## LLM-Driven Multi-Agent Systems for Autoformalization

Autoformalization systems can be developed within an agentic framework. In this tutorial, we use **MASA** ([Zhang et al., 2025](https://aclanthology.org/2025.emnlp-demos.44/)), presented at **EMNLP 2025 (System Demonstrations)**, as a running example to demonstrate how LLM-driven multi-agent systems can be constructed for autoformalization.

The overall architecture of MASA is illustrated below.

![MASA Framework](https://raw.githubusercontent.com/lanzhang128/multi_agent_autoformalization/main/multi_agent.png)

*Figure 2: Overview of the MASA framework.*

**Reference:** Zhang, L., Valentino, M., & Freitas, A. (2025). [MASA: LLM-Driven Multi-Agent Systems for Autoformalization](https://aclanthology.org/2025.emnlp-demos.44/). *Proceedings of EMNLP 2025: System Demonstrations*, 615–624.

### Setting Up the Environment for the Isabelle Theorem Prover

In [1]:
%cd /kaggle/working
!pip install -q gdown
!gdown 1c0dW3zLd0eUU5PC0sEorZ6V1Gec410W-
!tar -xzf /kaggle/working/Isabelle2024_linux.tar.gz -C /kaggle/working/
print('Extraction finished.')
import os
os.environ['ISABELLE_DIRPATH'] = os.path.abspath('/kaggle/working/Isabelle2024')

import nest_asyncio
nest_asyncio.apply()

/kaggle/working
Downloading...
From (original): https://drive.google.com/uc?id=1c0dW3zLd0eUU5PC0sEorZ6V1Gec410W-
From (redirected): https://drive.google.com/uc?id=1c0dW3zLd0eUU5PC0sEorZ6V1Gec410W-&confirm=t&uuid=c55f3091-f389-4a2e-9e47-7733a221b171
To: /kaggle/working/Isabelle2024_linux.tar.gz
100%|████████████████████████████████████████| 691M/691M [00:08<00:00, 85.2MB/s]
Extraction finished.


### Setting Up the Environment for the Lean4 Theorem Prover

In [2]:
%cd /kaggle/working
!wget https://github.com/leanprover/elan/releases/latest/download/elan-x86_64-unknown-linux-gnu.tar.gz
!tar -xzf elan-x86_64-unknown-linux-gnu.tar.gz
!./elan-init -y --no-modify-path --default-toolchain leanprover/lean4:stable
import os
os.environ["PATH"] += ":/root/.elan/bin"
os.environ["ELAN_HOME"] = "/root/.elan/"

!git clone https://github.com/leanprover-community/repl.git
%cd repl
!echo '' >> lakefile.toml
!echo '[[require]]' >> lakefile.toml
!echo 'name = "mathlib"' >> lakefile.toml
!echo 'scope = "leanprover-community"' >> lakefile.toml
!lake build repl
!lake update
!sed -i '1i import Mathlib\n' test/file.lean
!echo '{"path": "test/file.lean"}' | lake exe repl

import os
os.environ['REPL_PATH'] = os.path.abspath('/kaggle/working/repl')

/kaggle/working
--2026-10-27 13:38:37--  https://github.com/leanprover/elan/releases/latest/download/elan-x86_64-unknown-linux-gnu.tar.gz
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/leanprover/elan/releases/download/v4.2.4/elan-x86_64-unknown-linux-gnu.tar.gz [following]
--2026-10-27 13:38:37--  https://github.com/leanprover/elan/releases/download/v4.2.4/elan-x86_64-unknown-linux-gnu.tar.gz
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/128052474/c69badfe-a344-4969-82ff-82ceb4d0f604?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-27T14%3A19%3A03Z&rscd=attachment%3B+filename%3Delan-x86_64-unknown-linux-gnu.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-9

### Setting Up the MASA Repository

In [3]:
%cd /kaggle/working
!pip install rank_bm25
!git clone https://github.com/lanzhang128/multi_agent_autoformalization.git
%cd multi_agent_autoformalization
!mkdir test_results
!chmod +x theorem_prover/repl.sh

/kaggle/working
Cloning into 'multi_agent_autoformalization'...
remote: Enumerating objects: 1259, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 1259 (delta 54), reused 18 (delta 18), pack-reused 1166 (from 2)
Receiving objects: 100% (1259/1259), 9.00 MiB | 15.75 MiB/s, done.
Resolving deltas: 100% (187/187), done.
/kaggle/working/multi_agent_autoformalization


In [4]:
api_key = 'openai-api-key'

## Demonstration

### Example: Definition Autoformalization with Agents

We begin with a simple mathematical definition to demonstrate how LLM-based agents can translate informal mathematical descriptions into formal representations in **Lean**.

**Definition of Bradley--Terry model:** Given a pair of items $i$ and $j$ drawn from some population, the Bradley--Terry model estimates the probability that the pairwise comparison turns out true, as
$$\Pr(i > j) = \frac{p_i}{p_i + p_j}$$
where $p_i$ is a positive real-valued score assigned to individual $i$.

In [5]:
definition = "**Definition of Bradley--Terry model:** Given a pair of items $i$ and $j$ drawn from some population, the Bradley--Terry model estimates the probability that the pairwise comparison turns out true, as\n$$\\Pr(i > j) = \\frac{p_i}{p_i + p_j}$$\nwhere $p_i$ is a positive real-valued score assigned to individual $i$."

### LLMs

We begin by initializing the large language models (LLMs) used throughout this tutorial: **GPT-6.1-sol** and **Qwen-2.5-7B-Instruct**. These models will serve as the foundation for the autoformalization, critique, and refinement components introduced in the following sections.

In [6]:
from llm import OpenAILLM
gpt = OpenAILLM(
    api_key=api_key,
    model='gpt-6.1-sol',
    max_tokens=10000)

gpt-6.1-sol: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


In [7]:
from llm import HuggingFaceLLM
qwen = HuggingFaceLLM(model_id='Qwen/Qwen2.5-7B-Instruct')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### Zero-Shot Autoformalization Agent

We begin by constructing a **zero-shot autoformalization agent** within the **MASA framework**. Given a mathematical content expressed in natural language, the agent uses an LLM to generate its corresponding formal representation in a formal language, without relying on additional examples or external feedback. This serves as the baseline component for the subsequent critique and refinement stages.

In [8]:
from llm import BaseLLM
from agent.abstract import BaseAgent
from agent.agent_utils import extract_formal_code


class ZeroShotAgent(BaseAgent):
    def __init__(self, llm: BaseLLM, name: str = 'AutoformalizationAgent', formal_language: str = 'Lean4'):
        super().__init__(name=name)
        basic_system_prompt = (
            'You are an expert in formal language {formal_language}.\n'
            'You will be given a mathematical content written in natural language and LaTeX symbols.\n'
            'Your task is to provide the formal code of the given natural language mathematical content in {formal_language} with the following instructions:\n'
            '1. If the given mathematical content is a definition, the formal code does not require a proof.\n'
            '2. If the given mathematical content does not contain a proof, you should omit the proof and use the default \'sorry\' mode in the formal code.'
            '3. In case that you need to import any necessary preambles, you should not import any fake (non-exist) preambles.\n'
            '4. Ensure that the generated formal code is complete and syntactically well-formed, including all required language-specific declarations, delimiters, and closing constructs.\n'
            '5. You should wrap the formal code in a way illustrated as the following:\n'
            '%%%%%%%%%%\n'
            'Your Formal Code\n'
            '%%%%%%%%%%\n'
            'Strictly follow the instructions that have been claimed.\n')
        user_prompt = (
            'Natural language mathematical content: {{informal}}\n'
            'Give me the {formal_language} formal code of the content:\n')
        
        self.llm = llm
        self.formal_language = formal_language
        self.system_prompt = basic_system_prompt.replace('{formal_language}', self.formal_language)
        self.user_prompt = user_prompt.replace('{formal_language}', self.formal_language)

    def get_messages(self, informal: str = ''):
        messages = [{'role': 'system', 'content': self.system_prompt}]
        messages.append({'role': 'user', 'content': self.user_prompt.replace('{informal}', informal)})
        return messages

    def _agent_function(self, informal):
        messages = self.get_messages(informal=informal)
        response = self.llm.generate(messages)
        return extract_formal_code(response), response

Let's start by seeing how **Qwen2.5-7B-Instruct** handles this definition in a zero-shot setting.

In [9]:
agent_zs_qwen = ZeroShotAgent(llm=qwen, formal_language='Lean4')
code_qwen, _ = agent_zs_qwen(informal=definition)
print('code:\n', code_qwen)

code:
 import data.real.basic

namespace BradleyTerryModel

variable {I : Type} [decidable_eq I] -- Define a type I for items with decidable equality

/-- Definition of the Bradley--Terry model probability -/
def prob_i_gt_j (p : I → ℝ+) (i j : I) : ℝ := p i / (p i + p j)

end BradleyTerryModel



### Hard Critique Agent

Next, we use the built-in **hard critique agent**, which uses the **Lean theorem prover** as its backend, to check whether the generated formalization is syntactically valid and type-correct. If any errors are detected, the agent returns diagnostic feedback from Lean to help identify the issues.

If the generated Lean code does not include any library imports, the hard critique agent automatically prepends `import Mathlib` to ensure that the necessary mathematical definitions and theorems are available.

In [10]:
from agent import HardCritiqueAgent

agent_lean = HardCritiqueAgent(formal_language='Lean4', file_dir='./test_results')

correctness, error_details = agent_lean(formalization=code_qwen, file_prefix='lean_qwen_0')

with open('./test_results/lean_qwen_0.lean', 'r', encoding='utf-8') as f:
  qwen_formalization = f.read()
print('formalization:\n', qwen_formalization)
print('correctness:', correctness)
print('errors:\n', error_details)

logging file is at: ./test_results/lean.log
formalization:
 import data.real.basic

namespace BradleyTerryModel

variable {I : Type} [decidable_eq I] -- Define a type I for items with decidable equality

/-- Definition of the Bradley--Terry model probability -/
def prob_i_gt_j (p : I → ℝ+) (i j : I) : ℝ := p i / (p i + p j)

end BradleyTerryModel

correctness: False
errors:
 Identified error on line: 5
Error message: invalid binder annotation, type is not a class instance
  ?m.2

Note: Use the command `set_option checkBinderAnnotations false` to disable the check

Identified error on line: 8
Error message: unexpected token '+'; expected ')'




Next, let's see how **GPT-6.1-sol**, a more advanced LLM, handles the same definition in a zero-shot setting.

In [11]:
agent_zs_gpt = ZeroShotAgent(llm=gpt, formal_language='Lean4')
code_gpt, _ = agent_zs_gpt(informal=definition)
print('code:\n', code_gpt)

code:
 import Mathlib.Data.Real.Basic

/-- A Bradley–Terry model assigns a positive real score to each item. -/
structure BradleyTerryModel (Item : Type*) where
  score : Item → ℝ
  score_pos : ∀ i, 0 < score i

/-- The probability that item `i` wins a pairwise comparison with item `j`. -/
def BradleyTerryModel.winProbability {Item : Type*}
    (model : BradleyTerryModel Item) (i j : Item) : ℝ :=
  model.score i / (model.score i + model.score j)



In [12]:
correctness, error_details = agent_lean(formalization=code_gpt, file_prefix='lean_gpt_0')

with open('./test_results/lean_gpt_0.lean', 'r', encoding='utf-8') as f:
  gpt_formalization = f.read()
print('formalization:\n', gpt_formalization)
print('correctness:', correctness)
print('errors:\n', error_details)

formalization:
 import Mathlib.Data.Real.Basic

/-- A Bradley–Terry model assigns a positive real score to each item. -/
structure BradleyTerryModel (Item : Type*) where
  score : Item → ℝ
  score_pos : ∀ i, 0 < score i

/-- The probability that item `i` wins a pairwise comparison with item `j`. -/
def BradleyTerryModel.winProbability {Item : Type*}
    (model : BradleyTerryModel Item) (i j : Item) : ℝ :=
  model.score i / (model.score i + model.score j)

correctness: False
errors:
 Identified error on line: 9
Error message: failed to compile definition, consider marking it as 'noncomputable' because it depends on 'Real.instDivInvMonoid', which is 'noncomputable'




### Refine with Feedback Agent

The formalization generated by **GPT-6.1-sol** contains some errors, as identified by the hard critique agent. We can use these error messages as feedback to guide the LLM in correcting a formalization.

To do this, we define a **refinement agent** that takes the original formalization and the feedback as input, then generates a revised version. This allows us to iteratively improve the formalization based on feedback from different sources.

In [13]:
class RefineWithFeedbackAgent(BaseAgent):
    def __init__(self,
                 llm: BaseLLM,
                 name: str = 'RefineWithFeedbackAgent',
                 formal_language: str = 'Lean4'):
        super().__init__(name=name)

        basic_system_prompt = (
            'You are an expert in formal language {formal_language}.\n'
            'You will be given a mathematical content written in natural language and LaTeX symbols.\n'
            'You will also be given a formal code which attempted to describe the given mathematical content in {formal_language}.\n'
            'Some feedback about the formal code will also be given.\n'
            'Your task is to refine the given formal code to make it correct while maintaining the alignment with the given natural language mathematical content.\n'
            'Here are some instructions for your task:\n'
            '1. If the given mathematical content is a definition, the formal code does not require a proof.\n'
            '2. If the given mathematical content does not contain a proof, you should omit the proof and use the default \'sorry\' mode in the formal code.'
            '3. In case that you need to import any necessary preambles, you should not import any fake (non-exist) preambles.\n'
            '4. Ensure that the generated formal code is complete and syntactically well-formed, including all required language-specific declarations, delimiters, and closing constructs.\n'
            '5. You should wrap the formal code in a way illustrated as the following:\n'
            '%%%%%%%%%%\n'
            'Your Formal Code\n'
            '%%%%%%%%%%\n'
            'Strictly follow the instructions that have been claimed.\n')

        basic_user_prompt = (
            'Natural language mathematical content: {{informal}}\n'
            'There are some {formal_language} formal codes describing the given mathematical content: {{formal}}\n'
            'You should refine the formal code for your task to make it correct.\n'
            'Here are some feedback about the formal code which can be used to help your task: {{feedback}}\n')

        self.llm = llm

        self.formal_language = formal_language
        self.system_prompt = basic_system_prompt.replace('{formal_language}', self.formal_language)
        self.basic_user_prompt = basic_user_prompt.replace('{formal_language}', self.formal_language)

    def get_messages(self,
                     informal: str = '',
                     formal: str = '',
                     feedback: str = ''):
        messages = [{'role': 'system', 'content': self.system_prompt}]
        user_content = self.basic_user_prompt.replace('{informal}', informal).replace('{formal}', formal).replace('{feedback}', feedback)
        messages.append({'role': 'user', 'content': user_content})
        return messages

    def _agent_function(self,
                        informal: str = '',
                        formal: str = '',
                        feedback: str = ''):
        messages = self.get_messages(informal=informal,
                                     formal=formal, 
                                     feedback=feedback)
        response = self.llm.generate(messages)
        return extract_formal_code(response), response

Next, we prepare the error feedback and pass it, along with the original formalization, to the refinement agent to generate an improved version.

In [14]:
def error_feedback(correctness, error_details):
    temp = '\n'.join(error_details)

    if len(temp) > 10000:
        temp = temp[:10000]
    elif len(temp) == 0:
        temp = 'The theorem prover did not provide any error details.'

    feedback = (f'According to the theorem prover, the correctness of the provided formal code is {correctness}.\n'
                f'The error details are:\n{temp}\n\n')
    return feedback


agent_refine_gpt = RefineWithFeedbackAgent(llm=gpt, formal_language='Lean4')
code_refine, _ = agent_refine_gpt(informal=definition, formal=gpt_formalization, feedback=error_feedback(correctness, error_details))
correctness, error_details = agent_lean(formalization=code_refine, file_prefix='lean_refine_0')

with open('./test_results/lean_refine_0.lean', 'r', encoding='utf-8') as f:
  refine_formalization = f.read()
print('formalization:\n', refine_formalization)
print('correctness:', correctness)
print('errors:\n', error_details)

formalization:
 import Mathlib.Data.Real.Basic

/-- A Bradley–Terry model assigns a positive real score to each item. -/
structure BradleyTerryModel (Item : Type*) where
  score : Item → ℝ
  score_pos : ∀ i, 0 < score i

/-- The probability that item `i` wins a pairwise comparison with item `j`. -/
noncomputable def BradleyTerryModel.winProbability {Item : Type*}
    (model : BradleyTerryModel Item) (i j : Item) : ℝ :=
  model.score i / (model.score i + model.score j)

correctness: True
errors:
 


### Soft Critique Agent

The formalization above is now syntactically correct, but does it accurately capture the meaning of the original definition?

To evaluate this, we introduce a **soft critique agent** based on **LLM-as-a-Judge**. Unlike the hard critique agent, which checks syntactic and type correctness using Lean, the soft critique agent can assess the **semantic alignment** between the generated formalization and the original natural language definition. It provides a judgment along with an explanation to help identify potential semantic issues.

In [15]:
from agent.agent_utils import extract_judgement

class JudgeAgent(BaseAgent):
    def __init__(self,
                 llm: BaseLLM,
                 name: str = 'JudgeAgent',
                 formal_language: str = 'Lean4'):
        super().__init__(name=name)
        basic_system_prompt = (
        'You are an expert in formal language {formal_language}.\n'
        'You will be given a mathematical content written in natural language and LaTeX symbols.\n'
        'You will also be given a formal code which attempted to describe the given content in {formal_language}.\n'
        'Your task is to evaluate a specific aspect of the formal code.\n'
        'The description of the aspect is: {{aspect_description}}\n'
        'Your need to give two things about your evaluation:\n'
        '1. the judgement of whether the formalization meets this aspect. '
        'This should be a binary value in \"True\" or \"False\".\n'
        '2. the detailed explanation of your judgement.\n'
        'You should wrap your final results in a way illustrated as the following:\n'
        '%%%%%%%%%%\n'
        'Explanation: Your Detailed Explanation\n'
        'Judgement: Your Binary Judgement\n'
        '%%%%%%%%%%\n'
        'Strictly follow the instructions that have been claimed.\n')
        
        user_prompt = (
            'Natural language mathematical content: {{informal}}\n'
            'There are some {formal_language} formal codes describing the given mathematical content: {{formal}}\n'
        )

        self.llm = llm

        self.formal_language = formal_language
        self.basic_system_prompt = basic_system_prompt.replace('{formal_language}', self.formal_language)
        self.basic_user_prompt = user_prompt.replace('{formal_language}', self.formal_language)

    def get_messages(self,
                     informal: str = '',
                     formal: str = '',
                     aspect_description: str = ''):
        messages = [{'role': 'system', 'content': self.basic_system_prompt.replace('{aspect_description}', aspect_description)}]
        user_content = self.basic_user_prompt.replace('{informal}', informal).replace('{formal}', formal)
        messages.append({'role': 'user', 'content': user_content})
        return messages

    def _agent_function(self,
                        informal: str = '',
                        formal: str = '',
                        aspect_description: str = ''):
        messages = self.get_messages(informal=informal,
                                     formal=formal, 
                                     aspect_description=aspect_description)
        response = self.llm.generate(messages)
        return extract_judgement(response), response

Using the soft critique agent defined above, we can evaluate the quality of the generated formalization along two dimensions:

1. **Alignment Faithfulness:** Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?

2. **Formalization Correctness:** Is the generated formalization mathematically valid, natural, and well-formed in the target formal language?

These two dimensions help us assess not only whether the formalization preserves the intended meaning, but also whether it is expressed appropriately in Lean.

In [16]:
asp_af = 'Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?'
asp_fc = 'Is the generated formalization mathematically valid, natural, and well-formed in the target formal language?'

In [17]:
def print_judgments(judgments):
    for asp in judgments.keys():
        print('Aspect: ', asp)
        print(judgments[asp])
        print()

agent_judge_gpt = JudgeAgent(llm=gpt, formal_language='Lean4')

refine_judgments = {}
for asp in [asp_af, asp_fc]:
    judgment, _ = agent_judge_gpt(informal=definition, formal=refine_formalization, aspect_description=asp)
    refine_judgments[asp] = judgment

print_judgments(refine_judgments)

Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The formalization accurately captures the definition. `BradleyTerryModel` assigns each item a real-valued score through `score : Item → ℝ`, and `score_pos` requires every score to be strictly positive. The definition `winProbability model i j` is precisely \(p_i/(p_i+p_j)\), representing the probability that item \(i\) wins against item \(j\).

Strict positivity ensures that the denominator is nonzero and that the resulting value lies strictly between 0 and 1. Although the code does not explicitly prove these properties or construct an underlying probability space, neither is necessary to capture the stated model definition.
Judgement: True


Aspect:  Is the generated formalization mathematically valid, natural, and well-formed in the target formal language?
Explanation: The formalization is mathematically faithful and well-formed Lea

The soft critique agent judges the refined **GPT-6.1-sol** formalization positively on both dimensions. However, the formalization does not explicitly represent the probability $\Pr(i > j)$, indicating that some semantic misalignment remains. This highlights an important limitation: **LLM-based judges are not always reliable**, and their evaluations should not be treated as definitive.

Next, let's evaluate the zero-shot formalization generated by **Qwen2.5-7B-Instruct** using the same two dimensions.

In [18]:
qwen_judgments = {}
for asp in [asp_af, asp_fc]:
    judgment, _ = agent_judge_gpt(informal=definition, formal=qwen_formalization, aspect_description=asp)
    qwen_judgments[asp] = judgment
    
print_judgments(qwen_judgments)

Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The expression `p i / (p i + p j)` matches the intended Bradley–Terry formula, and indexing the scores by an item type `I` is appropriate. However, the score type `ℝ+` is not a standard Lean4/mathlib notation for strictly positive real numbers, and the code does not define it. Consequently, the formalization does not actually specify the required domain of positive real-valued scores.

This positivity requirement is essential: it ensures the denominator is positive and the resulting value is a valid probability. A faithful formalization could instead use `p : I → ℝ` with an assumption `∀ i, 0 < p i`, or explicitly define a subtype of strictly positive reals and coerce its elements to `ℝ` in the formula.

Thus, although the displayed formula is correct in intent, the code as supplied does not successfully formalize the definition.
Judg

The soft critique agent gives negative judgments on both dimensions, suggesting issues with both alignment faithfulness and formalization correctness. We can now turn these judgments and their explanations into feedback, then pass them, along with the original **Qwen2.5-7B-Instruct** formalization, to the previously defined **refinement agent** to improve the formalization.

In [19]:
import re

def judge_value(text):
    judgement = re.findall('Judgement: (True|False)', text, flags=re.DOTALL)
    if judgement:
        correctness = judgement[-1].split()[-1]
        if correctness == 'True':
            return True
        else:
            return False
    else:
        return None


def judge_feedback(judgments):
    feedback = ''
    for asp in judgments:
        if not judge_value(judgments[asp]):
            feedback += f'Aspect: {asp}\nEvaluation: {judgments[asp]}\n\n'
    return feedback

code_refine_qwen, _ = agent_refine_gpt(informal=definition, formal=qwen_formalization, feedback=judge_feedback(qwen_judgments))
correctness, error_details = agent_lean(formalization=code_refine_qwen, file_prefix='lean_refine_1')

with open('./test_results/lean_refine_1.lean', 'r', encoding='utf-8') as f:
  refine_qwen_formalization = f.read()
print('formalization:\n', refine_qwen_formalization)
print('correctness:', correctness)
print('errors:\n', error_details)

formalization:
 import Mathlib.Data.Real.Basic

namespace BradleyTerryModel

variable {I : Type*}

/-- The Bradley–Terry probability that item `i` beats item `j`,
given a strictly positive real score for each item. -/
def prob_i_gt_j (p : I → {x : ℝ // 0 < x}) (i j : I) : ℝ :=
  (p i : ℝ) / ((p i : ℝ) + (p j : ℝ))

end BradleyTerryModel

correctness: False
errors:
 Identified error on line: 9
Error message: failed to compile definition, consider marking it as 'noncomputable' because it depends on 'Real.instDivInvMonoid', which is 'noncomputable'




Comparing the refined formalizations, we observe that when **GPT-6.1-sol** refines the zero-shot formalization generated by **Qwen2.5-7B-Instruct**, it tends to preserve the original structure rather than rewrite the code from scratch.

This illustrates an interesting characteristic of **feedback-guided refinement**: the refinement agent can build upon the initial formalization and make changes based on the provided feedback, rather than generating an entirely new formalization. Since the same mathematical statement can have multiple valid formalizations, preserving the original structure is not necessarily good or bad. It simply shows how the initial candidate can influence the refinement process.

In [20]:
print('Refined GPT formalization:\n', refine_formalization)
print('Refined Qwen formalization:\n', refine_qwen_formalization)

Refined GPT formalization:
 import Mathlib.Data.Real.Basic

/-- A Bradley–Terry model assigns a positive real score to each item. -/
structure BradleyTerryModel (Item : Type*) where
  score : Item → ℝ
  score_pos : ∀ i, 0 < score i

/-- The probability that item `i` wins a pairwise comparison with item `j`. -/
noncomputable def BradleyTerryModel.winProbability {Item : Type*}
    (model : BradleyTerryModel Item) (i j : Item) : ℝ :=
  model.score i / (model.score i + model.score j)

Refined Qwen formalization:
 import Mathlib.Data.Real.Basic

namespace BradleyTerryModel

variable {I : Type*}

/-- The Bradley–Terry probability that item `i` beats item `j`,
given a strictly positive real score for each item. -/
def prob_i_gt_j (p : I → {x : ℝ // 0 < x}) (i j : I) : ℝ :=
  (p i : ℝ) / ((p i : ℝ) + (p j : ℝ))

end BradleyTerryModel



### Building an End-to-End Workflow with Agents

With the agents defined above, we can now combine them into an **end-to-end autoformalization workflow**. The workflow integrates zero-shot generation, hard critique from the theorem prover, soft critique from LLM judges, and feedback-guided refinement.

The workflow proceeds as follows:

1. **Zero-Shot Autoformalization:** We start by generating an initial formalization from the natural language mathematical statement.

2. **Hard Critique:** The generated formalization is checked by the theorem prover (Lean). If any syntax or type errors are detected, the error messages are collected as feedback and passed to the refinement agent.

3. **Soft Critique:** Once the formalization passes the theorem prover check, we use the LLM judge to evaluate its **alignment faithfulness** and **formalization correctness**. If either evaluation identifies issues, the judgments and explanations are used as feedback for refinement.

4. **Feedback-Guided Refinement:** The refinement agent takes the current formalization and the collected feedback to generate an updated version. The updated formalization is then checked again, starting with the hard critique.

This process continues until the formalization passes both the hard and soft critiques, or the maximum number of LLM calls is reached. Note that this budget includes calls for zero-shot generation, LLM-based judgments, and refinement, while theorem prover checks do not count toward the limit.

The workflow demonstrates how individual agents can be connected to form an **iterative autoformalization system**, where feedback from both the theorem prover and LLM judges guides the refinement process.

In [21]:
class AutoformalizationWorkflow:
    def __init__(self,
                 llm: BaseLLM, 
                 formal_language='Lean4', 
                 max_LLM_call=10):
        self.formal_language = formal_language
        if self.formal_language[:8] == 'Isabelle':
            self.extension = 'thy'
        elif self.formal_language == 'Lean4':
            self.extension = 'lean'
        else:
            raise ValueError(f'{formal_language} is not supported.')
        
        self.agent_zs = ZeroShotAgent(llm=llm, formal_language=formal_language)
        self.agent_tp = HardCritiqueAgent(formal_language=formal_language, file_dir='./test_results')
        self.agent_refine = RefineWithFeedbackAgent(llm=llm, formal_language=formal_language)
        self.agent_judge = JudgeAgent(llm=llm, formal_language=formal_language)
        self.aspects = [asp_af, asp_fc]
        self.max_LLM_call = 10
    
    def run(self, informal: str = ''):
        code, _ = self.agent_zs(informal=informal)
        num_LLM_call = 1
        correctness, error_details = self.agent_tp(formalization=code, file_prefix=f'test_0')
        with open(f'./test_results/test_0.{self.extension}', 'r', encoding='utf-8') as f:
            formalization = f.read()
        judgments = {}

        i = 1
        while num_LLM_call < self.max_LLM_call:
            if correctness == 'False':
                feedback = error_feedback(correctness, error_details)
            else:
                for asp in self.aspects:
                    judgment, _ = self.agent_judge(informal=informal, formal=formalization, aspect_description=asp)
                    num_LLM_call += 1
                    judgments[asp] = judgment
                    if num_LLM_call == self.max_LLM_call:
                        break
                feedback = judge_feedback(judgments)

            if not feedback and len(judgments) == len(self.aspects):
                print(f'Autoformalization finished with {num_LLM_call} LLM calls.')
                return formalization, correctness, error_details, judgments
            else:
                if num_LLM_call == self.max_LLM_call:
                    print(f'Autoformalization not finished with {num_LLM_call} LLM calls.')
                    return formalization, correctness, error_details, judgments
                else:
                    code, _ = self.agent_refine(informal=informal, formal=formalization, feedback=feedback)
                    num_LLM_call += 1
                    correctness, error_details = self.agent_tp(formalization=code, file_prefix=f'test_{i}')
                    with open(f'./test_results/test_{i}.{self.extension}', 'r', encoding='utf-8') as f:
                        formalization = f.read()
                    judgments = {}
                    i += 1
                    
        print(f'Autoformalization not finished with {num_LLM_call} LLM calls.')
        return formalization, correctness, error_details, judgments

Let's first run the workflow on the **Bradley–Terry model definition** using **Qwen2.5-7B-Instruct** as the backend LLM and examine the final result after feedback-guided refinement.

In [22]:
qwen_workflow = AutoformalizationWorkflow(qwen)
formalization, correctness, error_details, judgments = qwen_workflow.run(informal=definition)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

logging file is at: ./test_results/lean.log
Autoformalization not finished with 10 LLM calls.
formalization:
 import data.real.basic

namespace BradleyTerryModel

variables {I : Type} [decidable_eq I] -- Define a type I for items with decidable equality

/-- Definition of the Bradley--Terry model probability -/
def prob_i_gt_j (p : I → ℝ+) (i j : I) : ℝ := p i / (p i + p j)

end BradleyTerryModel
```

Explanation of changes:
1. **Line 5**: No change needed as the syntax is correct.
2. **Line 8**: Corrected the addition operator `+` to ensure it is properly used.
3. **Line 11**: Removed the unnecessary double quotes `"` and ensured the correct use of parentheses and operators.
4. **Line 14**: No change needed as the syntax is correct.
5. **Line 17**: No change needed as the syntax is correct.

This code is now syntactically correct and aligns with the provided mathematical content.<|im_end|>
correctness: False
errors:
 Identified error on line: 5
Error message: unexpected identifier; ex

The final formalization still contains some issues, suggesting that **Qwen2.5-7B-Instruct** may struggle with this autoformalization task, even with feedback-guided refinement.

Next, let's run the same workflow on the definition using **GPT-6.1-sol** as the backend LLM and examine the final result.

In [23]:
gpt_workflow = AutoformalizationWorkflow(gpt)
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=definition)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

logging file is at: ./test_results/lean.log
Autoformalization finished with 4 LLM calls.
formalization:
 import Mathlib

/-- The Bradley–Terry probability that item `i` beats item `j`,
given a strictly positive real score for each item. -/
noncomputable def bradleyTerryProbability {Item : Type*}
    (score : Item → {p : ℝ // 0 < p}) (i j : Item) : ℝ :=
  (score i).val / ((score i).val + (score j).val)

correctness: True
errors:
 
Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The formalization accurately represents the Bradley–Terry model. The type `Item` represents the population of items, and `score : Item → {p : ℝ // 0 < p}` assigns each item a strictly positive real-valued score. For items `i` and `j`, the definition returns `(score i).val / ((score i).val + (score j).val)`, exactly matching \(p_i/(p_i+p_j)\). Strict positivity ensures that the denominator is positive and that

This time, the workflow successfully produces a formalization that passes both the hard and soft critiques after **4 LLM calls**.

### Example: Theorem Statement Autoformalization

Next, let's use our autoformalization workflow with **GPT-6.1-sol** as the backend LLM to formalize the following theorem statement.

**Classical Goldbach's Conjecture:** For every even integer $n > 2$, there exist two prime numbers $p$ and $q$ such that
$$n = p + q.$$

In [24]:
statement = "**Classical Goldbach's Conjecture:** For every even integer $n > 2$, there exist two prime numbers $p$ and $q$ such that\n$$n = p + q.$$"
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=statement)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

Autoformalization finished with 3 LLM calls.
formalization:
 import Mathlib

theorem classical_goldbach_conjecture :
    ∀ n : ℤ, Even n → 2 < n →
      ∃ p q : ℕ, Nat.Prime p ∧ Nat.Prime q ∧
        n = (p : ℤ) + (q : ℤ) := by
  sorry

correctness: True
errors:
 
Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The formalization accurately captures the conjecture. It quantifies over every integer `n : ℤ`, with hypotheses `Even n` and `2 < n`, corresponding exactly to an even integer greater than 2. It then asserts the existence of two natural numbers `p` and `q`, both satisfying `Nat.Prime`, whose integer casts sum to `n`. Using natural numbers for the primes correctly represents the usual positive prime numbers. There is no requirement that the two primes be distinct, consistent with the original statement.

The `sorry` leaves the conjecture unproved, but it does not affect the ac

This time, the workflow successfully produces a formalization that passes both the hard and soft critiques after **3 LLM calls**.

### Example: Full-Theorem Autoformalization

Next, let's extend our autoformalization workflow to **full-theorem autoformalization**, where we aim to formalize both the theorem statement and its proof.

We use **GPT-6.1-sol** as the backend LLM to demonstrate how LLM-based mathematical reasoning agents can autoformalize and formally prove the following mathematical theorem.

**Theorem (An Equivalent Formulation of Goldbach's Conjecture):** An equivalent statement of the Goldbach conjecture is that for every positive integer $m$, there exist prime numbers $p$ and $q$ such that

$$\varphi(p)+\varphi(q)=2m,$$

where $\varphi$ denotes Euler's totient function.

**Proof:**

Recall that for every prime number $p$, Euler's totient function satisfies $\varphi(p)=p-1.$

**($\Rightarrow$)** Assume that the classical Goldbach conjecture holds.

Let $m$ be any positive integer. Since $2m+2$ is an even integer greater than $2$, the Goldbach conjecture guarantees the existence of primes $p$ and $q$ such that

$$p+q=2m+2.$$

Using the identity $\varphi(p)=p-1$ for primes, we obtain

$$\varphi(p)+\varphi(q)=(p-1)+(q-1)=p+q-2=2m.$$

Thus, the totient formulation holds.

**($\Leftarrow$)** Conversely, assume that the totient formulation holds.

Let $N$ be any even integer greater than $2$. Then there exists a positive integer $m$ such that

$$N=2m+2.$$

By the totient formulation, there exist primes $p$ and $q$ satisfying

$$\varphi(p)+\varphi(q)=2m.$$

Since $p$ and $q$ are prime, we have

$$p+q=(\varphi(p)+1)+(\varphi(q)+1)=\varphi(p)+\varphi(q)+2=2m+2=N.$$

Therefore, every even integer greater than $2$ is the sum of two primes, which is precisely the classical Goldbach conjecture.

Hence, the two formulations are logically equivalent. $\square$


In [25]:
full_theorem = "**Theorem (An Equivalent Formulation of Goldbach's Conjecture):** An equivalent statement of the Goldbach conjecture is that for every positive integer $m$, there exist prime numbers $p$ and $q$ such that\n\n$$\\varphi(p)+\\varphi(q)=2m,$$\n\nwhere $\\varphi$ denotes Euler's totient function.\n\n**Proof:**\n\nRecall that for every prime number $p$, Euler's totient function satisfies $\\varphi(p)=p-1.$\n\n**($\\Rightarrow$)** Assume that the classical Goldbach conjecture holds.\n\nLet $m$ be any positive integer. Since $2m+2$ is an even integer greater than $2$, the Goldbach conjecture guarantees the existence of primes $p$ and $q$ such that\n\n$$p+q=2m+2.$$\n\nUsing the identity $\\varphi(p)=p-1$ for primes, we obtain\n\n$$\\varphi(p)+\\varphi(q)=(p-1)+(q-1)=p+q-2=2m.$$\n\nThus, the totient formulation holds.\n\n**($\\Leftarrow$)** Conversely, assume that the totient formulation holds.\n\nLet $N$ be any even integer greater than $2$. Then there exists a positive integer $m$ such that\n\n$$N=2m+2.$$\n\nBy the totient formulation, there exist primes $p$ and $q$ satisfying\n\n$$\\varphi(p)+\\varphi(q)=2m.$$\n\nSince $p$ and $q$ are prime, we have\n\n$$p+q=(\\varphi(p)+1)+(\\varphi(q)+1)=\\varphi(p)+\\varphi(q)+2=2m+2=N.$$\n\nTherefore, every even integer greater than $2$ is the sum of two primes, which is precisely the classical Goldbach conjecture.\n\nHence, the two formulations are logically equivalent. $\\square$"
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=full_theorem)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

Autoformalization finished with 3 LLM calls.
formalization:
 import Mathlib

namespace GoldbachEquivalence

def GoldbachConjecture : Prop :=
  ∀ N : ℕ, 2 < N → Even N →
    ∃ p q : ℕ, Nat.Prime p ∧ Nat.Prime q ∧ p + q = N

def TotientGoldbachConjecture : Prop :=
  ∀ m : ℕ, 0 < m →
    ∃ p q : ℕ,
      Nat.Prime p ∧ Nat.Prime q ∧
        Nat.totient p + Nat.totient q = 2 * m

theorem goldbach_iff_totient_goldbach :
    GoldbachConjecture ↔ TotientGoldbachConjecture := by
  constructor
  · intro hGoldbach m hm
    have hEven : Even (2 * m + 2) := by
      exact ⟨m + 1, by omega⟩
    obtain ⟨p, q, hp, hq, hsum⟩ :=
      hGoldbach (2 * m + 2) (by omega) hEven
    refine ⟨p, q, hp, hq, ?_⟩
    rw [Nat.totient_prime hp, Nat.totient_prime hq]
    have hp₂ : 2 ≤ p := hp.two_le
    have hq₂ : 2 ≤ q := hq.two_le
    omega
  · intro hTotient N hN hEven
    obtain ⟨k, hk⟩ := hEven
    have hm : 0 < k - 1 := by omega
    obtain ⟨p, q, hp, hq, hsum⟩ := hTotient (k - 1) hm
    refine ⟨p, q, hp, hq, ?

The workflow again successfully produces a formalization that passes both the hard and soft critiques after **3 LLM calls** as well.

### Example: Cross-Prover Autoformalization

So far, we have demonstrated our autoformalization workflow using **Lean** as the theorem prover. Now, let's switch to **Isabelle/HOL** while keeping the same agentic workflow and **GPT-6.1-sol** as the backend LLM.

This example demonstrates how the same workflow can be adapted to different theorem provers by changing the target formal language and its corresponding verification backend.

In [26]:
gpt_workflow = AutoformalizationWorkflow(gpt, formal_language='Isabelle/HOL')
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=definition)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

logging file is at: ./test_results/isabelle.log
Isabelle server started with info: server "test" = 127.0.0.1:36747 (password "baedde00-742a-4943-926d-6b6ce265ace8")
 in 2.63s.
Isabelle server session 6acaf663-a72a-4980-aabd-089b237166a4 started in 7.79s.
Isabelle server use_theory ended in 2.35s.
Autoformalization finished with 3 LLM calls.
formalization:
 theory test_0
imports Complex_Main
begin

locale bradley_terry_model =
  fixes score :: "'a ⇒ real"
  assumes score_positive: "⋀i. 0 < score i"
begin

definition win_probability :: "'a ⇒ 'a ⇒ real" where
  "win_probability i j = score i / (score i + score j)"

end

end
correctness: True
errors:
 
Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The formalization accurately captures the stated Bradley–Terry model. The type variable `'a` represents the population of items, and `score :: "'a ⇒ real"` assigns a real-valued score to ev

In [27]:
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=statement)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

Isabelle server use_theory timeout with 120s.
Isabelle server 127.0.0.1:36747 shutdown.
Isabelle server started with info: server "test" = 127.0.0.1:37061 (password "cfdd125a-e2dd-430e-b68b-cc61823a6a37")
 in 1.62s.
Isabelle server session db243f83-70e9-4c5e-949d-de0ffc0c1818 started in 6.36s.
Not Finished.
Isabelle server use_theory ended in 61.39s.
Autoformalization finished with 4 LLM calls.
formalization:
 theory test_1
imports "HOL-Computational_Algebra.Primes"
begin

theorem classical_goldbach_conjecture:
  fixes n :: int
  assumes "even n" and "n > 2"
  shows "\<exists>p q :: nat. prime p \<and> prime q \<and> n = int p + int q"
  sorry

end
correctness: True
errors:
 
Aspect:  Does the generated formalization accurately capture the intended mathematical meaning of the original natural language mathematics?
Explanation: The formalization accurately states the conjecture. It fixes an arbitrary integer `n` and assumes precisely that `n` is even and greater than 2. Its conclusion a

In [28]:
formalization, correctness, error_details, judgments = gpt_workflow.run(informal=full_theorem)
print('formalization:\n', formalization)
print('correctness:', correctness)
print('errors:\n', error_details)
print_judgments(judgments)

Isabelle server use_theory timeout with 120s.
Isabelle server 127.0.0.1:37061 shutdown.
Isabelle server started with info: server "test" = 127.0.0.1:45885 (password "89da5d6c-731c-4dee-b090-186525e6634b")
 in 1.70s.
Isabelle server session 8968bb7b-6164-4c22-bcb5-c9abae5bea63 started in 6.52s.
Not Finished.
Isabelle server use_theory timeout with 120s.
Isabelle server 127.0.0.1:45885 shutdown.
Isabelle server started with info: server "test" = 127.0.0.1:36563 (password "026456bd-9826-4807-9a8f-9dcfda24a05a")
 in 1.65s.
Isabelle server session 1d9e8b35-5ea8-4f0c-929d-6d79f44773dd started in 6.81s.
Not Finished.
Isabelle server use_theory timeout with 120s.
Isabelle server 127.0.0.1:36563 shutdown.
Isabelle server started with info: server "test" = 127.0.0.1:38293 (password "72052f67-71f9-43fe-8241-8fef7b4fc8f3")
 in 1.72s.
Isabelle server session 31eb5bac-8b97-4a3d-9cd3-c8150e64b330 started in 6.91s.
Not Finished.
Isabelle server use_theory timeout with 120s.
Isabelle server 127.0.0.1:3

In this cross-prover demonstration, we observe that **GPT-6.1-sol** struggles more with autoformalization in **Isabelle/HOL** than in **Lean 4**, despite using the same agentic workflow.

There are several possible reasons for this:

- **Model Familiarity:** The LLM may be more familiar with Lean 4 than Isabelle/HOL, including its syntax, mathematical libraries, and formalization conventions.
- **Theorem Prover Feedback:** Different theorem provers provide different error messages, which may affect how effectively the refinement agent can identify and correct errors.
- **Language-Specific Challenges:** Even with the same agentic workflow, generating valid formalizations may require different strategies for different theorem provers.

However, **cross-prover autoformalization** remains important for the formal verification of mathematics. The validity of a mathematical claim should not depend on a particular theorem prover.

In particular, verifying the same mathematical claim across multiple theorem provers offers two important benefits:

- **Independent Verification:** Verifying a mathematical claim in different theorem provers, such as **Lean** and **Isabelle/HOL**, can increase confidence in its correctness by reducing reliance on a single formal system.
- **Semantic Faithfulness:** A theorem prover can verify that a formal statement is provable, but it cannot guarantee that the statement faithfully captures the intended mathematical meaning. Independently formalizing and verifying the same claim in different theorem provers can reduce the risk of shared semantic misalignment, especially when the formalizations are developed independently.

**Key Takeaway:** Cross-prover autoformalization is not just about supporting different formal languages. It also provides an opportunity to strengthen confidence in both the **formal correctness** and **semantic faithfulness** of autoformalized mathematics.

In [29]:
gpt_workflow.agent_tp.theorem_prover.terminate()

Isabelle server 127.0.0.1:45561 shutdown.
